# M3L3 E01 — Clasificador de intenciones (Resolution)
### Módulo 3 · Lecture 3 · Sistemas Multiagente

## ¿Qué vas a aprender hoy?
- clasificar la consulta antes de responder.
- Conectar el concepto con M3L2.
- Leer código pequeño con explicación previa.
- Interpretar resultados y checks.


## ¿Qué necesitás saber antes?

Venís de M3L2 con LangChain, LCEL, PromptTemplate, RAG con FAISS y memoria conversacional. En M3L3 usamos esas piezas para coordinar varios agentes.

> **Sistema multiagente:** arquitectura donde varias unidades especializadas colaboran bajo una política de coordinación.


## Instalación e imports

En un notebook productivo podrías instalar `langchain`, `langchain-openai` y `faiss-cpu`. Aquí usamos Python estándar para que el foco sea el diseño multiagente y no la API key.


In [ ]:
from typing import Callable, TypedDict, Literal
from dataclasses import dataclass, field
import json

print("Setup listo: usamos Python estándar para que el notebook pueda correr sin API key.")


## Sección 1 — La intención de una consulta

> **Intent:** propósito operativo de la consulta. No responde al usuario; decide a qué agente enviar la pregunta.

| Intent | Descripción | Ejemplo |
|---|---|---|
| `hr` | RR.HH. | vacaciones |
| `tech` | soporte técnico | VPN |
| `billing` | facturación | reembolso |
| `unknown` | ambiguo o fuera de alcance | almuerzo |


Definimos los valores permitidos. Este contrato evita que el sistema enrute a nombres inventados por el modelo.


In [ ]:
VALID_INTENTS = ["hr", "tech", "billing", "unknown"]
VALID_CONFIDENCE = ["high", "medium", "low"]


## Sección 2 — Clasificador y parser JSON

En producción, el clasificador sería un `ChatPromptTemplate | llm | StrOutputParser`. Aquí simulamos el LLM para enfocarnos en el contrato.

`parse_intent(raw_output)` convierte texto JSON en un dict seguro.


In [ ]:
def mock_classifier_llm(query: str) -> str:
    text = query.lower()
    if any(w in text for w in ["vacaciones", "seguro", "beneficio", "licencia"]):
        return json.dumps({"intent": "hr", "confidence": "high"})
    if any(w in text for w in ["vpn", "contraseña", "mfa", "notebook"]):
        return json.dumps({"intent": "tech", "confidence": "high"})
    if any(w in text for w in ["factura", "reembolso", "pago", "recibo"]):
        return json.dumps({"intent": "billing", "confidence": "high"})
    return json.dumps({"intent": "unknown", "confidence": "low"})

def parse_intent(raw_output: str) -> dict:
    try:
        parsed = json.loads(raw_output)
    except json.JSONDecodeError:
        return {"intent": "unknown", "confidence": "low"}
    intent = parsed.get("intent", "unknown")
    confidence = parsed.get("confidence", "low")
    return {"intent": intent if intent in VALID_INTENTS else "unknown", "confidence": confidence if confidence in VALID_CONFIDENCE else "low"}

def classify_intent(query: str) -> dict:
    return parse_intent(mock_classifier_llm(query)) | {"query": query}


## Sección 3 — Pruebas

Probamos ejemplos de cada dominio y un caso fuera de alcance. Observá que el clasificador no responde la pregunta: solo decide el destino.


In [ ]:
for query in ["¿Cómo pido vacaciones?", "No puedo entrar a la VPN", "Quiero cargar una factura", "¿Qué hay de almuerzo?"]:
    print(classify_intent(query))


## Checks automáticos

Los checks verifican el contrato mínimo del ejercicio. En Starter pueden fallar hasta completar los TODOs; en Resolution deben pasar.


In [ ]:
def run_checks():
    assert classify_intent("vacaciones")["intent"] == "hr"
    assert classify_intent("vpn")["intent"] == "tech"
    assert classify_intent("factura")["intent"] == "billing"
    assert classify_intent("tema raro")["intent"] == "unknown"
    print("Checks E01 OK")
run_checks()


## ¿Qué aprendiste hoy?

- Clasificar la consulta antes de responder.
- Separar responsabilidades vuelve el sistema más auditable.
- Los contratos explícitos hacen que el orquestador dependa menos de texto libre.

## Próximo ejercicio

Continuá con el siguiente notebook de M3L3 para agregar una pieza más de coordinación multiagente.
